In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq 
from typing import Annotated # LangGraph uses Annotated to understand how state should behave 
from typing_extensions import TypedDict #  lets you define a dictionary with fixed keys and fixed value types.
from langgraph.graph import StateGraph,START,END # A flowchart that passes a shared state between steps
from langgraph.graph.message import add_messages # To handle chat messages in state  


load_dotenv() 
os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]="CourseLanggraph"

groq_api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=groq_api_key,model='llama-3.1-8b-instant')


class State(TypedDict): # This defines the shape of the shared state that flows through your graph. 
    # Every node in this graph will receive and return a dictionary that looks like this. 
    messages:Annotated[list,add_messages] # List to store chat messages 

# Typically this list holds: Human messages ,AI messages,Tool messages
# Whenever a node returns new messages, append them to the existing list instead of replacing it. 

graph_builder=StateGraph(State)   



In [ ]:

from langchain_community.utilities import ArxivAPIWrapper,WikipediaAPIWrapper
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=300) 
arxiv_tool=ArxivQueryRun(api_wrapper=arxiv_wrapper)

wiki_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=300)
wiki_tool=wiki_tool=WikipediaQueryRun(api_wrapper=wiki_wrapper)

In [ ]:
tools=[wiki_tool,arxiv_tool]

In [ ]:
llm_with_tools=llm.bind_tools(tools=tools)

def chatbot(state:State):
  return {"messages":llm_with_tools.invoke(state['messages'])}

In [ ]:
from langgraph.prebuilt import ToolNode,tools_condition

graph_builder.add_node("chatbot",chatbot) 

tool_node=ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START,"chatbot")

graph=graph_builder.compile()


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
user_input="Tell me about ,Transformers paper"

events=graph.stream(
     {"messages": [("user", user_input)]},stream_mode="values"
)

for event in events:
  event["messages"][-1].pretty_print()